# 06 — Verificar, congelar e registrar promoção experimental

## Objetivo
Empacotar um candidato já selecionado pelo NB05. Verificar recarga, identidade, preprocessing e contrato de inferência; registrar elegibilidade pelo protocolo v2.

## Entradas
Diretório explícito de seleção v2; runs, plano e critérios vinculados; certificado de dados.

## Saídas
`models/experimental_v2/frozen/<freeze_id>/` e `reports/experimental_v2/promotions/<freeze_id>.json`.

## Permissões de dados
Métricas persistidas de validation e imagem de validation para smoke/paridade. Calibration permanece not_applied. Sem leitura de holdout/final test.

## Reprodutibilidade
Freeze por conteúdo: checkpoint, config, classes, preprocess, seed, dados, seleção, critérios e código. Mudanças de conteúdo produzem outra identidade.

Protocolo: [experimental_protocol_v2.md](../docs/experimental_protocol_v2.md).
Os resultados anteriores estão catalogados como `legacy_v1`; não comparar seus scores diretamente aos v2.


In [ ]:
from pathlib import Path
import sys

# Bootstrap de importação; a resolução dos dados fica nos módulos compartilhados.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/experimental_protocol.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.v2_artifacts import repository_root, read_json, run_id
ROOT = repository_root()

from scripts.v2_artifacts import relative_path
from scripts.v2_data import dataset_context, experiment_identity
from scripts.v2_release import verified_selection, smoke_run, freeze_candidate, promote_frozen
SELECTION_DIRECTORY = None  # copiar o caminho relativo produzido pelo NB05
EXECUTE_FREEZE = False
display(experiment_identity(dataset_context(ROOT)))
selection_dir = relative_path(ROOT, SELECTION_DIRECTORY) if SELECTION_DIRECTORY else None
candidate_dir = None
if selection_dir is not None:
    selection, plan, candidate_dir = verified_selection(ROOT, selection_dir)
    display(selection)
else:
    print("Pendente: seleção CNN v2 completa e critérios registrados. Não importar o candidato operacional v1.")


## Validar antes de congelar
Verificar hashes do pacote, config versus checkpoint, ordem das classes, logits versus IDs e seleção versus todas as seeds previstas.
O smoke recarrega o checkpoint sem baixar pesos, confere inferência individual versus logits de validation e verifica erro para imagem inválida.
Não executa avaliação de desempenho em uma nova partição.


In [ ]:
smoke = None
if candidate_dir is not None:
    smoke = smoke_run(ROOT, candidate_dir)
    display(smoke)
else:
    print("Smoke real pendente de candidato; o comportamento das funções é coberto por testes sintéticos.")


## Freeze e gate v2
O freeze inclui `calibration.status=not_applied` e `policy=none`. Nenhum threshold é inventado.
O gate usa a média de macro-F1 de validation da receita, recalls do candidato, critérios pré-registrados, integridade, smoke e freeze.
Campos adicionais ou métricas de outra partição não entram no contrato de elegibilidade.

Promoção aqui significa **elegibilidade experimental registrada**. Não altera `models/classification/active_model.json` nem o modelo servido.
Integração operacional exige uma etapa de implantação própria; não deve ser um efeito colateral de Run All.


In [ ]:
freeze_id = None
promotion = None
if EXECUTE_FREEZE:
    if selection_dir is None:
        raise ValueError("Informe a seleção v2 completa antes do freeze")
    freeze_id = freeze_candidate(ROOT, selection_dir)
    promotion = promote_frozen(ROOT, freeze_id)
    display({"freeze_id": freeze_id, "promotion": promotion})
else:
    print("Freeze/promoção não executados. Sem candidato ou critérios completos, a decisão permanece pendente.")


## Conclusão
A elegibilidade é determinada antes de qualquer leitura do holdout. O pacote congelado é verificado por hash a cada uso; alterações não podem conservar a mesma identidade válida.

## Artefatos produzidos
Freeze, pacote, smoke, model card e decisão imutável de promoção experimental quando explicitamente habilitados. Resultados/modelo operacional legacy_v1 permanecem preservados.

## Próxima etapa
Avaliação pós-freeze pelo script separado e NB07. O resultado dessa avaliação não pode selecionar outro candidato nem retroagir sobre o gate.
